![image.png](https://i.imgur.com/a3uAqnb.png)

# English-Japanese Seq2Seq Translation Model

![Neural Machine Translation](https://miro.medium.com/v2/resize:fit:1400/1*sO-SP58T4brE9EHazHSeGA.png)

## **⚠️ Important Note on Training Approach**

**This notebook demonstrates Seq2Seq architecture for English-Japanese translation, but uses a simplified training approach for educational purposes:**

- ✅ **Purpose**: Showcase Seq2Seq implementation and Japanese NLP challenges
- ⚠️ **Limitation**: Only uses training set (no validation/test split)
- 🎯 **Result**: Model will overfit, but this demonstrates the complexity of Japanese translation
- 📚 **Learning Goal**: Understand architecture, not production-ready model

**In production, you would need proper train/validation/test splits and regularization techniques!**

## **📌 Sequence-to-Sequence (Seq2Seq) Model**

A **Seq2Seq model** consists of two main components:
1. **Encoder**: Processes input sequence (English) into fixed-size context vector
2. **Decoder**: Generates output sequence (Japanese) from context vector

### **🔹 Key Features**
- **Variable-length inputs/outputs**: Can handle sequences of different lengths
- **Attention mechanism**: (Not implemented here, but commonly used, see next lab)
- **Teacher forcing**: Uses ground truth during training for faster convergence

### **🔹 Architecture Overview**

## 1️⃣ Dataset Loading and Initial Exploration


In [7]:
import kagglehub
import pandas as pd
import os
from collections import Counter
import re

# Download and load the Korean parallel sentences dataset
path = kagglehub.dataset_download("bryanpark/parallelsents")
csv_file = [f for f in os.listdir(path) if f.endswith('.csv')][0]
dataset_path = os.path.join(path, csv_file)

# Load CSV with error handling
df = pd.read_csv(dataset_path, on_bad_lines='skip')

# Filter valid translation pairs and convert to dictionary format
valid_pairs = df.dropna(subset=['EXAMPLE (EN)', 'EXAMPLE (JA)'])
dataset = {
    'en': valid_pairs['EXAMPLE (EN)'].tolist(),
    'ja': valid_pairs['EXAMPLE (JA)'].tolist()
}
dataset

{'en': ['The store opens at 10 A.M.',
  "Do you like it? What's the price?",
  'My house is near my school.',
  'I sometimes drink beer.',
  'Come tomorrow, if possible.',
  'My father went to Seoul early in the morning.',
  'I taught her how to drive.',
  'Have you looked inside your bag?',
  'Oil is lighter than water.',
  'My favorite singer is Yongpil Jo.',
  'My chest was hurting since last night.',
  'Sit there, in the middle.',
  'It is good to read books in the fall.',
  'I like skiing the most.',
  'Almost every home is hooked up to the internet now.',
  'Did you bring towels?',
  'The whole family has caught a cold.',
  'They sell various kinds of things at the shop.',
  'She has a beautiful voice.',
  'The other three classes are one hour each.',
  'I wanted to be a nurse when I was younger.',
  'Do you like ribs?',
  'I think I have a cold.',
  'Thank you.',
  'I got a strange feeling when I saw him.',
  "It's getting cold suddenly.",
  'The price of vegetables rose this ye

**📝 What's happening here:**
- Loading a parallel English-Japanese corpus from Korean language learning materials
- This dataset contains 1000 most frequent Korean words with example sentences translated into multiple languages
- We're using the English examples as source and Japanese translations as target
- Japanese is a logographic language with complex writing systems (hiragana, katakana, kanji)
- 日本語は複雑な言語です (Japanese is a complex language, no spaces, and each word is a letter)

## 2️⃣ Vocabulary Analysis and Tokenization


In [10]:
def simple_tokenize(text):
    """
    Simple tokenization function that:
    1. Removes punctuation
    2. Splits on whitespace
    Note: For Japanese, proper tokenization would use tools like MeCab!
    """
    cleaned = re.sub(r'[^\w\s]', ' ', text)
    return cleaned.split()

# Build vocabularies
english_words = []
japanese_words = []
for text in dataset['en']:
    english_words.extend(simple_tokenize(str(text).lower()))
for text in dataset['ja']:
    # For Japanese, we'll treat each character as a token for simplicity
    japanese_words.extend(list(str(text)))

english_vocab = Counter(english_words)
japanese_vocab = Counter(japanese_words)
print(f"Japanese vocab: {len(japanese_vocab):,} unique words")
print(f"English vocab: {len(english_vocab):,} unique words")

Japanese vocab: 990 unique words
English vocab: 1,553 unique words


**📝 Key Observations:**
- Japanese has fewer unique characters than English words, but each character carries more meaning
- Japanese uses multiple writing systems: hiragana, katakana, and kanji
- This simple character-level tokenization is basic but functional for demonstration

**🔹 Japanese NLP Challenges:**
1. **No spaces between words** (word segmentation is difficult)
2. **Multiple writing systems** (hiragana, katakana, kanji mixed together)
3. **Complex grammar** (SOV word order, particles, honorifics)
4. **Context-dependent meaning** (same characters can have different readings)

## 3️⃣ Vocabulary Reduction and Mapping


In [12]:
import pickle

ENGLISH_VOCAB_SIZE = 3000
JAPANESE_VOCAB_SIZE = 2000
SPECIAL_TOKENS = ['<PAD>', '<SOS>', '<EOS>', '<UNK>']

def create_vocab_mapping(vocab_dict, special_tokens):
    """
    Creates bidirectional mappings between words and indices.
    
    Special tokens:
    - <PAD>: Padding for variable-length sequences
    - <SOS>: Start of sequence marker
    - <EOS>: End of sequence marker  
    - <UNK>: Unknown words (out of vocabulary)
    """
    word2idx = {}
    idx2word = {}
    
    # Add special tokens first (indices 0-3)
    for i, token in enumerate(special_tokens):
        word2idx[token] = i
        idx2word[i] = token
    
    # Add most frequent words
    for i, word in enumerate(vocab_dict.keys(), len(special_tokens)):
        word2idx[word] = i
        idx2word[i] = word
    
    return word2idx, idx2word

# Create reduced vocabularies (keep only most frequent words)
english_vocab_reduced = dict(english_vocab.most_common(ENGLISH_VOCAB_SIZE))
japanese_vocab_reduced = dict(japanese_vocab.most_common(JAPANESE_VOCAB_SIZE))

en_word2idx, en_idx2word = create_vocab_mapping(english_vocab_reduced, SPECIAL_TOKENS)
ja_word2idx, ja_idx2word = create_vocab_mapping(japanese_vocab_reduced, SPECIAL_TOKENS)

print(f"Reduced vocabularies: English={len(en_word2idx)}, Japanese={len(ja_word2idx)}")

Reduced vocabularies: English=1557, Japanese=994


**📝 Why vocabulary reduction?**
- **Memory efficiency**: Smaller embedding tables
- **Training speed**: Fewer parameters to update
- **Generalization**: Focus on most important characters/words
- **Trade-off**: Some characters become `<UNK>` tokens
- **Japanese-specific**: Character-level approach reduces out-of-vocabulary issues

## 4️⃣ Data Preprocessing and Sequence Conversion


In [13]:
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

def clean_and_tokenize(text, is_japanese=False):
    """
    Clean and tokenize text:
    1. Remove bracketed annotations [...]
    2. Remove punctuation (for English) or keep Japanese characters
    3. Convert to lowercase (English only)
    4. Split into tokens (words for English, characters for Japanese)
    """
    text = re.sub(r'\[.*?\]', '', text)  # Remove annotations
    
    if is_japanese:
        # For Japanese, keep all characters and split by character
        text = re.sub(r'[^\u3040-\u309F\u30A0-\u30FF\u4E00-\u9FAF\u3000-\u303F]', '', text)  # Keep Japanese chars
        return list(text.strip()) if text.strip() else []
    else:
        # For English, remove punctuation and split by words
        text = re.sub(r'[^\w\s]', ' ', text)  # Remove punctuation
        text = text.strip().lower() if text.strip() else ''
        return text.split() if text else []

def text_to_sequence(text, word2idx, max_length=None, is_japanese=False):
    """
    Convert text to sequence of token indices:
    1. Add <SOS> token at start
    2. Convert words/characters to indices (use <UNK> for unknown tokens)
    3. Add <EOS> token at end
    4. Truncate if exceeds max_length
    """
    tokens = clean_and_tokenize(text, is_japanese)
    sequence = [word2idx.get('<SOS>', 1)]  # Start token
    
    for token in tokens:
        idx = word2idx.get(token, word2idx.get('<UNK>', 3))
        sequence.append(idx)
    
    sequence.append(word2idx.get('<EOS>', 2))  # End token
    
    # Truncate if too long
    if max_length and len(sequence) > max_length:
        sequence = sequence[:max_length-1] + [word2idx.get('<EOS>', 2)]
    
    return sequence


**📝 Sequence format:**
```
Original English: "Hello everyone"
Tokenized: ["hello", "everyone"]
Sequence: [<SOS>, hello_idx, everyone_idx, <EOS>]
Indices: [1, 145, 892, 2] - probably not, but just for show
```

```
Original Japanese:: "私は日本語は話せませんが、アニメをよく見るので、このラボは冗談でやりました（笑）"

Tokenized: ["私", "は", "日", "本", "語", "は", "話", "せ", "ま", "せ", "ん", "が", "、", "ア", "ニ", "メ", "を", "よ", "く", "見", "る", "の", "で", "、", "こ", "の", "ラ", "ボ", "は", "冗", "談", "で", "や", "り", "ま", "し", "た", "（", "笑", "）"]

Sequence: [<SOS>, 私_idx, は_idx, 日_idx, 本_idx, 語_idx, は_idx, 話_idx, せ_idx, ま_idx, せ_idx, ん_idx, が_idx, 、_idx, ア_idx, ニ_idx, メ_idx, を_idx, よ_idx, く_idx, 見_idx, る_idx, の_idx, で_idx, 、_idx, こ_idx, の_idx, ラ_idx, ボ_idx, は_idx, 冗_idx, 談_idx, で_idx, や_idx, り_idx, ま_idx, し_idx, た_idx, （_idx, 笑_idx, ）_idx, <EOS>]

Indices : [1, 305, 78, 112, 256, 341, 78, 452, 210, 187, 210, 23, 91, 9, 512, 513, 514, 67, 412, 390, 278, 421, 333, 299, 9, 145, 333, 520, 521, 78, 612, 613, 299, 701, 702, 187, 356, 204, 900, 950, 901, 2]
 - probably not, but just for show
```

## 5️⃣ Dataset Creation and DataLoader


In [30]:
SUBSET_SIZE = 1000

subset_data = {
    'en': dataset['en'][:SUBSET_SIZE],
    'ja': dataset['ja'][:SUBSET_SIZE]
}

class TranslationDataset(Dataset):
    """
    PyTorch Dataset for English-Japanese translation pairs.
    Filters out sequences that are too short/long for stable training.
    """
    def __init__(self, data, en_word2idx, ja_word2idx, max_en_len=50, max_ja_len=80):
        self.samples = []
        
        for i in range(len(data['en'])):
            en_seq = text_to_sequence(data['en'][i], en_word2idx, max_en_len, is_japanese=False)
            ja_seq = text_to_sequence(data['ja'][i], ja_word2idx, max_ja_len, is_japanese=True)
            
            # Filter: keep sequences with reasonable length
            if 3 <= len(en_seq) <= max_en_len and 3 <= len(ja_seq) <= max_ja_len:
                self.samples.append((en_seq, ja_seq))
        
        print(f"Dataset: {len(data['en'])} → {len(self.samples)} samples")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        en_seq, ja_seq = self.samples[idx]
        return {
            'english': torch.tensor(en_seq, dtype=torch.long),
            'japanese': torch.tensor(ja_seq, dtype=torch.long)
        }

def collate_fn(batch):
    """
    Custom collate function to handle variable-length sequences.
    Pads sequences to the same length within each batch.
    """
    english_seqs = [item['english'] for item in batch]
    japanese_seqs = [item['japanese'] for item in batch]
    
    # Pad sequences (padding_value=0 corresponds to <PAD> token)
    english_padded = pad_sequence(english_seqs, batch_first=True, padding_value=0)
    japanese_padded = pad_sequence(japanese_seqs, batch_first=True, padding_value=0)
    
    return {'english': english_padded, 'japanese': japanese_padded}

train_dataset = TranslationDataset(subset_data, en_word2idx, ja_word2idx)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, collate_fn=collate_fn)

Dataset: 998 → 998 samples


**🚨 Training Data Issue:**
- We're using the same data for training that we'll test on
- This will lead to **severe overfitting**
- Model will memorize rather than generalize
- **For demonstration purposes only!**
- **Small dataset**: Only ~999 samples makes overfitting even worse

## 6️⃣ Seq2Seq Model Architecture


In [16]:
import torch.nn as nn

class Encoder(nn.Module):
    """
    LSTM-based encoder that processes English input sequence.
    
    Flow: English tokens → Embeddings → LSTM → Hidden states
    Returns final hidden and cell states as context for decoder.
    """
    def __init__(self, vocab_size, embed_size, hidden_size, num_layers=2):
        super(Encoder, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size, padding_idx=0)
        self.lstm = nn.LSTM(embed_size, hidden_size, num_layers, 
                           batch_first=True, dropout=0.3)
        
    def forward(self, x):
        # x shape: (batch_size, seq_len)
        embedded = self.embedding(x)  # (batch_size, seq_len, embed_size)
        outputs, (hidden, cell) = self.lstm(embedded)
        # hidden, cell: (num_layers, batch_size, hidden_size)
        return hidden, cell

class Decoder(nn.Module):
    """
    LSTM-based decoder that generates Japanese output sequence.
    
    Uses teacher forcing during training:
    - Input: previous ground truth token
    - Output: prediction for next token
    """
    def __init__(self, vocab_size, embed_size, hidden_size, num_layers=2):
        super(Decoder, self).__init__()
        self.vocab_size = vocab_size
        self.embedding = nn.Embedding(vocab_size, embed_size, padding_idx=0)
        self.lstm = nn.LSTM(embed_size, hidden_size, num_layers, 
                           batch_first=True, dropout=0.3)
        self.fc = nn.Linear(hidden_size, vocab_size)  # Project to vocab size
        
    def forward(self, x, hidden, cell):
        # x shape: (batch_size, 1) - single token input
        embedded = self.embedding(x)  # (batch_size, 1, embed_size)
        output, (hidden, cell) = self.lstm(embedded, (hidden, cell))
        # output: (batch_size, 1, hidden_size)
        prediction = self.fc(output)  # (batch_size, 1, vocab_size)
        return prediction, hidden, cell

class Seq2Seq(nn.Module):
    """
    Complete Sequence-to-Sequence model combining encoder and decoder.
    
    Training process:
    1. Encode English sequence to context vector
    2. Decode context to Japanese sequence using teacher forcing
    """
    def __init__(self, encoder, decoder, device):
        super(Seq2Seq, self).__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
        
    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        """
        Forward pass with teacher forcing.
        
        Args:
            src: English input sequence (batch_size, src_len)
            trg: Japanese target sequence (batch_size, trg_len)
            teacher_forcing_ratio: Probability of using ground truth vs model prediction
        """
        batch_size = src.shape[0]
        trg_len = trg.shape[1]
        trg_vocab_size = self.decoder.vocab_size
        
        # Store all predictions
        outputs = torch.zeros(batch_size, trg_len, trg_vocab_size).to(self.device)
        
        # Encode source sequence
        hidden, cell = self.encoder(src)
        
        # First decoder input is <SOS> token
        input_token = trg[:, 0].unsqueeze(1)  # (batch_size, 1)
        
        # Generate sequence token by token
        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(input_token, hidden, cell)
            outputs[:, t] = output.squeeze(1)
            
            # Teacher forcing: use ground truth or model prediction
            use_teacher_forcing = torch.rand(1).item() < teacher_forcing_ratio
            top1 = output.argmax(2)  # Get predicted token
            input_token = trg[:, t].unsqueeze(1) if use_teacher_forcing else top1
            
        return outputs

# Create model components
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
encoder = Encoder(len(en_word2idx), 256, 512, 2)
decoder = Decoder(len(ja_word2idx), 256, 512, 2)
model = Seq2Seq(encoder, decoder, device).to(device)
print(f"Model created on {device}")

Model created on cuda


**📝 Architecture Details:**
- **Embedding dimension**: 256 (dense vector representation)
- **Hidden dimension**: 512 (LSTM internal state size)
- **Layers**: 2 (stacked LSTM layers)
- **Dropout**: 0.3 (regularization during training)

**🔹 Teacher Forcing:**
- During training: Use ground truth previous token
- During inference: Use model's own predictions
- Helps with training stability and speed

## 7️⃣ Training Setup and Loss Function


In [17]:
import torch.optim as optim
from torch.nn import CrossEntropyLoss
import time

# Loss function ignores padding tokens (index 0)
criterion = CrossEntropyLoss(ignore_index=0)
optimizer = optim.Adam(model.parameters(), lr=0.001)

def train_epoch(model, train_loader, criterion, optimizer, device):
    """
    Train model for one epoch.
    
    Process:
    1. Forward pass: English → Japanese prediction
    2. Compute loss: Compare prediction with ground truth
    3. Backward pass: Update model parameters
    """
    model.train()
    total_loss = 0
    
    for batch_idx, batch in enumerate(train_loader):
        src = batch['english'].to(device)     # English input
        trg = batch['japanese'].to(device)    # Japanese target
        
        optimizer.zero_grad()
        
        # Forward pass with teacher forcing
        output = model(src, trg, teacher_forcing_ratio=0.5)
        
        # Reshape for loss computation
        # output: (batch_size, trg_len-1, vocab_size)
        # trg: (batch_size, trg_len-1)
        output = output[:, 1:].reshape(-1, output.shape[-1])  # Skip <SOS> token
        trg = trg[:, 1:].reshape(-1)                          # Skip <SOS> token
        
        # Compute cross-entropy loss
        loss = criterion(output, trg)
        loss.backward()
        
        # Gradient clipping to prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        total_loss += loss.item()
    
    return total_loss / len(train_loader)


**📝 Loss Computation:**
- **CrossEntropyLoss**: Measures prediction accuracy
- **ignore_index=0**: Ignores `<PAD>` tokens in loss
- **Gradient clipping**: Prevents unstable training

## 8️⃣ Training Loop


In [ ]:
NUM_EPOCHS = 40

print("🚨 TRAINING NOTE:")
print("This model will OVERFIT because we're using only training data!")
print("Purpose: Demonstrate Seq2Seq challenges with Japanese translation")

🚨 TRAINING NOTE:
This model will OVERFIT because we're using only training data!
Purpose: Demonstrate Seq2Seq challenges with Japanese translation


In [24]:
for epoch in range(1, NUM_EPOCHS + 1):
    start_time = time.time()
    epoch_loss = train_epoch(model, train_loader, criterion, optimizer, device)
    
    print(f"Epoch {epoch}: Loss = {epoch_loss:.4f}, Time = {time.time() - start_time:.1f}s")
    
    # # Early stopping when loss gets reasonably low
    # if epoch_loss < 2.5:
    #     print("Target loss reached!")
    #     break

Epoch 1: Loss = 2.2838, Time = 1.9s
Epoch 2: Loss = 2.1238, Time = 1.5s
Epoch 3: Loss = 1.9033, Time = 1.5s
Epoch 4: Loss = 1.6372, Time = 1.5s
Epoch 5: Loss = 1.4352, Time = 1.6s
Epoch 6: Loss = 1.3131, Time = 1.5s
Epoch 7: Loss = 1.0297, Time = 1.6s
Epoch 8: Loss = 0.8464, Time = 1.6s
Epoch 9: Loss = 0.6334, Time = 1.6s
Epoch 10: Loss = 0.5093, Time = 1.6s
Epoch 11: Loss = 0.3605, Time = 1.6s
Epoch 12: Loss = 0.2771, Time = 1.6s
Epoch 13: Loss = 0.2006, Time = 1.5s
Epoch 14: Loss = 0.1561, Time = 1.5s
Epoch 15: Loss = 0.1136, Time = 1.6s
Epoch 16: Loss = 0.0872, Time = 1.5s
Epoch 17: Loss = 0.0681, Time = 1.6s
Epoch 18: Loss = 0.0534, Time = 1.6s
Epoch 19: Loss = 0.0502, Time = 1.6s
Epoch 20: Loss = 0.0449, Time = 1.6s


KeyboardInterrupt: 

## 9️⃣ Translation Function


In [25]:
# Cell 10: Translation Function
def translate_sentence(model, sentence, en_word2idx, ja_idx2word, device, max_length=100):
    """
    Translate a single English sentence to Japanese.
    
    Process:
    1. Convert English text to token indices
    2. Encode with trained encoder
    3. Decode step-by-step (greedy decoding)
    4. Convert indices back to Japanese characters
    """
    model.eval()
    
    with torch.no_grad():
        # Prepare source sequence
        src_seq = text_to_sequence(sentence, en_word2idx, max_length=50, is_japanese=False)
        src_tensor = torch.tensor(src_seq, dtype=torch.long).unsqueeze(0).to(device)
        
        # Encode source
        hidden, cell = model.encoder(src_tensor)
        
        # Start decoding with <SOS> token
        trg_indexes = [ja_word2idx.get('<SOS>', 1)]
        
        # Generate tokens one by one
        for _ in range(max_length):
            trg_tensor = torch.tensor([trg_indexes[-1]], dtype=torch.long).unsqueeze(0).to(device)
            output, hidden, cell = model.decoder(trg_tensor, hidden, cell)
            
            # Get most likely next token
            pred_token = output.argmax(2).item()
            trg_indexes.append(pred_token)
            
            # Stop if <EOS> token is generated
            if pred_token == ja_word2idx.get('<EOS>', 2):
                break
        
        # Convert indices to characters (skip <SOS> and <EOS>)
        trg_tokens = [ja_idx2word.get(i, '<UNK>') for i in trg_indexes[1:-1]]
        return ''.join(trg_tokens)  # Join without spaces for Japanese

**📝 Inference Process:**
- **Greedy decoding**: Always pick most likely token
- **Alternative**: Beam search for better results
- **No teacher forcing**: Use model's own predictions


## 🔟 Testing Translation


In [27]:
# Test translation
test_english = subset_data['en'][0]
translation = translate_sentence(model, test_english, en_word2idx, ja_idx2word, device)
print(f"English: {test_english}")
print(f"Translation: {translation}")

English: The store opens at 10 A.M.
Translation: その店は十時に開く。


**Expected Result:**
- Translation quality will be poor due to:
  1. **Overfitting** on small training set
  2. **Simple character-level tokenization** not handling Japanese word boundaries
  3. **No attention mechanism** losing context in long sequences
  4. **Mixed writing systems** making character prediction challenging

## 1️⃣1️⃣ BLEU Evaluation


In [28]:
from collections import Counter
import math

def compute_bleu_score(reference, candidate, max_n=4):
    """
    Compute BLEU score to measure translation quality.
    
    For Japanese, we'll use character-level n-grams since Japanese doesn't use spaces.
    Higher scores (0-1) indicate better translation quality.
    """
    def get_ngrams(tokens, n):
        if len(tokens) < n:
            return []
        return [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]
    
    # For Japanese, use character-level comparison
    ref_tokens = list(reference)  # Split into characters
    cand_tokens = list(candidate)  # Split into characters
    
    if len(cand_tokens) == 0:
        return 0.0
    
    # Compute precision for each n-gram level
    precisions = []
    for n in range(1, max_n + 1):
        ref_ngrams = Counter(get_ngrams(ref_tokens, n))
        cand_ngrams = Counter(get_ngrams(cand_tokens, n))
        
        if len(cand_ngrams) == 0:
            precisions.append(0.0)
            continue
            
        # Count matches
        matches = sum(min(count, ref_ngrams.get(ngram, 0)) 
                     for ngram, count in cand_ngrams.items())
        precision = matches / len(get_ngrams(cand_tokens, n))
        precisions.append(precision)
    
    # Brevity penalty
    ref_len = len(ref_tokens)
    cand_len = len(cand_tokens)
    bp = 1.0 if cand_len > ref_len else math.exp(1 - ref_len / cand_len) if cand_len > 0 else 0.0
    
    # Final BLEU score
    if all(p > 0 for p in precisions):
        bleu = bp * math.exp(sum(math.log(p) for p in precisions) / len(precisions))
    else:
        bleu = 0.0
    
    return bleu

def evaluate_bleu_by_length(model, subset_data, en_word2idx, ja_idx2word, device, num_samples=150):
    """
    Evaluate model performance on different sentence lengths.
    English-Japanese translation difficulty varies significantly with sentence length.
    """
    # Collect samples with their lengths
    samples_with_length = []
    for i in range(min(num_samples, len(subset_data['en']))):
        english_text = subset_data['en'][i]
        japanese_text = subset_data['ja'][i]
        text_length = len(english_text.split())
        samples_with_length.append((i, english_text, japanese_text, text_length))
    
    # Sort by length and categorize
    samples_with_length.sort(key=lambda x: x[3])
    lengths = [x[3] for x in samples_with_length]
    short_threshold = lengths[len(lengths)//3]
    long_threshold = lengths[2*len(lengths)//3]
    
    categories = {'short': [], 'medium': [], 'long': []}
    
    for sample in samples_with_length:
        idx, english, japanese, length = sample
        if length <= short_threshold:
            categories['short'].append(sample)
        elif length >= long_threshold:
            categories['long'].append(sample)
        else:
            categories['medium'].append(sample)
    
    # Evaluate each category
    results = {}
    for category_name, samples in categories.items():
        bleu_scores = []
        for idx, english_text, reference_japanese, length in samples:
            model_translation = translate_sentence(model, english_text, en_word2idx, ja_idx2word, device)
            bleu = compute_bleu_score(reference_japanese, model_translation)
            bleu_scores.append(bleu)
        
        if bleu_scores:
            avg_bleu = sum(bleu_scores) / len(bleu_scores)
            good_count = sum(1 for s in bleu_scores if s > 0.1)
            results[category_name] = {
                'avg_bleu': avg_bleu,
                'good_count': good_count,
                'total': len(bleu_scores)
            }
    
    # Print results
    for category in ['short', 'medium', 'long']:
        if category in results:
            r = results[category]
            print(f"{category.capitalize()}: BLEU = {r['avg_bleu']:.4f}, Good Rate = {r['good_count']/r['total']*100:.1f}%")
    
    return results

**📝 BLEU Score Interpretation:**
- **0.0-0.1**: Poor translation
- **0.1-0.3**: Understandable but low quality
- **0.3-0.5**: Good translation
- **0.5+**: Excellent translation
    - In my humble opinion, this score is useless :)

## 1️⃣2️⃣ Final Evaluation


In [29]:
# Run BLEU evaluation
print("📊 BLEU Score Evaluation (Remember: This is overfitted!)")
print("=" * 50)
bleu_results = evaluate_bleu_by_length(model, subset_data, en_word2idx, ja_idx2word, device)

📊 BLEU Score Evaluation (Remember: This is overfitted!)
Short: BLEU = 0.7451, Good Rate = 86.0%
Medium: BLEU = 0.9493, Good Rate = 100.0%
Long: BLEU = 0.9256, Good Rate = 100.0%


**Expected Results:**
- Scores will be artificially high due to overfitting
- Real-world performance would be much lower
- Demonstrates the challenge of Japanese NLP (without attention)
- Character-level BLEU may be misleading for Japanese text quality

## **🎯 Key Takeaways and Improvements**

### **❌ Current Issues:**
1. **No train/validation/test split** → Severe overfitting
2. **Simple character tokenization** → Poor Japanese word segmentation
3. **No attention mechanism** → Context loss in long sequences
4. **Small dataset** → Limited generalization (~1000 samples)
5. **No beam search** → Suboptimal decoding

### **✅ Production Improvements:**
1. **Proper data splitting** (80/10/10 train/val/test)
2. **Japanese-specific tokenization** (MeCab, SentencePiece, BPE)
3. **Attention mechanism** or **Transformer architecture**
4. **Larger dataset** with domain diversity
5. **Advanced decoding** (beam search, length penalties)
6. **Regularization techniques** (dropout, weight decay)

### **🔹 Japanese-Specific Challenges:**
- **No word boundaries**: こんにちはみなさん (no spaces between words)
- **Multiple writing systems**: ひらがな (hiragana), カタカナ (katakana), and 漢字 (kanji) mixed together
- **Honorific system**: Different forms for politeness levels 

Contributed by: A weeb